In [13]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import date, datetime, timedelta
# web scrapping
import bs4 as bs
import requests
import lxml
from functools import reduce
# matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from ipysigma import Sigma
from pyvis.network import Network
import requests
import re
from bs4 import BeautifulSoup
from io import StringIO
from dbconnection import MySQLDatabase
from utils import getSymbols, getData, get_last_date, get_marketid_simbols
import warnings
import itertools
warnings.filterwarnings("ignore", category=UserWarning, module="pandas")
sns.set_theme()

In [2]:
db = MySQLDatabase("financialmarkets")

In [11]:
def getSymbols(url):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
    }
    resp = requests.get(url, headers=headers)

    if resp.status_code != 200:
        raise ValueError(f"Error al obtener la página: {resp.status_code}")

    soup = BeautifulSoup(resp.text, "lxml")

    # Buscar cualquier tabla con clase 'wikitable'
    #table = soup.find("table", {"id": lambda x: x and "constituents" in x})
    table = soup.find("table", {"class": "wikitable"})
    #print(len(table[]))
    #print(table)
    if table is None:
        raise ValueError("No se encontró ninguna tabla con clase 'wikitable'")

    # Intentar identificar la tabla correcta: debe contener columna con "Company" o "Firma"
    
    # Convertir tabla a DataFrame
    #table = pd.read_html(StringIO(str(table)))[0]
    # Buscar la tabla principal
    #table = soup.find("table", {"class": "wikitable"})

    rows = table.find_all("tr")

    data = []

    for row in rows[1:]:
        cols = row.find_all("td")
        if len(cols) >= 4:
            code = cols[0].text.strip()
            currency = cols[3].text.strip()
            data.append([code, currency])

    df = pd.DataFrame(data, columns=["Code", "Currency"])

    return df

In [16]:
base = getSymbols('https://en.wikipedia.org/wiki/ISO_4217?utm_source=chatgpt.com')
base.head()

,Code,Num,D[a],Currency,Locations listed for this currency[b]
0,AED,784,2,United Arab Emirates dirham,United Arab Emirates
1,AFN,971,2,Afghan afghani,Afghanistan
2,ALL,8,2,Albanian lek,Albania
3,AMD,51,2,Armenian dram,Armenia
4,AOA,973,2,Angolan kwanza,Angola


In [33]:
pairs = list(itertools.permutations(base["Code"], 2))

df_pairs = pd.DataFrame(pairs, columns=["code1", "code2"])

# merge para currency1
df_pairs = df_pairs.merge(base, left_on="code1", right_on="Code", how="left")
df_pairs.rename(columns={"Currency": "currency1"}, inplace=True)

# merge para currency2
df_pairs = df_pairs.merge(base, left_on="code2", right_on="Code", how="left")
df_pairs.rename(columns={"Currency": "currency2"}, inplace=True)
df = df_pairs[['code1','currency1','code2','currency2']].copy()
df.loc[:,'symbol'] = [x+y+'=X' for x,y in zip(df['code1'],df['code2'])]
df.loc[:,'name'] = [x+'/'+y for x,y in zip(df['currency1'],df['currency2'])]
df

,code1,currency1,code2,currency2,symbol,name
0,AED,United Arab Emirates dirham,AFN,Afghan afghani,AEDAFN=X,United Arab Emirates dirham/Afghan afghani
1,AED,United Arab Emirates dirham,ALL,Albanian lek,AEDALL=X,United Arab Emirates dirham/Albanian lek
2,AED,United Arab Emirates dirham,AMD,Armenian dram,AEDAMD=X,United Arab Emirates dirham/Armenian dram
3,AED,United Arab Emirates dirham,AOA,Angolan kwanza,AEDAOA=X,United Arab Emirates dirham/Angolan kwanza
4,AED,United Arab Emirates dirham,ARS,Argentine peso,AEDARS=X,United Arab Emirates dirham/Argentine peso
...,...,...,...,...,...,...
31501,ZWG,Zimbabwe Gold,XUA,ADB Unit of Account,ZWGXUA=X,Zimbabwe Gold/ADB Unit of Account
31502,ZWG,Zimbabwe Gold,XXX,No currency,ZWGXXX=X,Zimbabwe Gold/No currency
31503,ZWG,Zimbabwe Gold,YER,Yemeni rial,ZWGYER=X,Zimbabwe Gold/Yemeni rial
31504,ZWG,Zimbabwe Gold,ZAR,South African rand,ZWGZAR=X,Zimbabwe Gold/South African rand


In [34]:
# generamos variables no existentes
# df['Ticker'] = [re.sub(r"\s+", "", x)+'.L' for x in df['Ticker'].astype('str')]
# df = df.rename(columns={'Ticker':'Symbol','Company':'Security', 'FTSE industry classification benchmark sector[37]':'GICS Sector'})
df['sector_name'] = ['currency' for x in df['symbol']]
df['sub_industry'] = ['SD' for x in df['symbol']]
df['cik'] = ['SD' for x in df['symbol']]
df['founded'] = [9999 for x in df['symbol']]
df['date_added'] = ['0000-00-00' for x in df['symbol']]
df['headquarters'] = ['0000-00-00' for x in df['symbol']]
# df = df[['Symbol','Security','GICS Sector','GICS Sub-Industry','Headquarters Location', 'Date added','CIK','Founded']]
# df.head()

**Ingresamos mercado**

In [23]:
# ---------------------------
# 3️Insertar/actualizar mercados
# ---------------------------
markets = pd.DataFrame({
    'market_name': ['FOREX'],
    'country': ['None'],
    'currency': ['None']
})
markets

,market_name,country,currency
0,FOREX,None,None


In [24]:
db.insert_to_db(markets, tabla="markets", batch_size=5000)

✅ Conexión exitosa


In [25]:
# Obtener market_id
market_id = db.execute_query("SELECT * FROM markets")
market_id

,market_id,market_name,country,currency
0,1,NASDAQ,USA,USD
1,2,S&P 500,USA,USD
2,3,IPC MX,MX,Peso
3,4,DAX,GERMAN,DEM
4,5,FTSE100,UKX,GBP
5,6,FOREX,None,None


In [29]:
market_id = 6

In [26]:
df.columns

Index(['code1', 'currency1', 'code2', 'currency2', 'symbol'], dtype='object')

In [35]:
df.loc[:,"market_id"] = [market_id for x in df['symbol']]
companies = df[["market_id",'symbol','name','sector_name','sub_industry','date_added','headquarters','cik','founded']]
#companies = companies.reset_index(drop=True)
companies.head()

,market_id,symbol,name,sector_name,sub_industry,date_added,headquarters,cik,founded
0,6,AEDAFN=X,United Arab Emirates dirham/Afghan afghani,currency,SD,0000-00-00,0000-00-00,SD,9999
1,6,AEDALL=X,United Arab Emirates dirham/Albanian lek,currency,SD,0000-00-00,0000-00-00,SD,9999
2,6,AEDAMD=X,United Arab Emirates dirham/Armenian dram,currency,SD,0000-00-00,0000-00-00,SD,9999
3,6,AEDAOA=X,United Arab Emirates dirham/Angolan kwanza,currency,SD,0000-00-00,0000-00-00,SD,9999
4,6,AEDARS=X,United Arab Emirates dirham/Argentine peso,currency,SD,0000-00-00,0000-00-00,SD,9999


In [36]:
db.insert_to_db(companies, tabla="companies", batch_size=100)

In [37]:
db.close()

🔒 Conexión cerrada
